### **1. Raw Data Ingestion**
Loads the raw Brazilian geolocation dataset into a PySpark DataFrame with automatic schema inference to initialize the spatial data engineering pipeline.

In [ ]:
# ── CELL 1: INITIALIZE SPARK SESSION ──────────────────────────────────────
import os
import pyspark
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Build SparkSession locally to avoid Python version mismatch with external cluster
spark = SparkSession.builder \
    .appName("olist-notebook-analysis") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to reduce noise inside the notebook
spark.sparkContext.setLogLevel("WARN")

print("Spark Session created successfully!")
print("Spark Master URI:", spark.sparkContext.master)

🎯 Spark Session created successfully!
Spark Master URI: local[*]


In [2]:
# ==========================================================================
# 1. LOAD GEOLOCATION DATASET FROM MINIO BRONZE LAYER
# ==========================================================================

# Load the geolocation dataset from the Bronze layer in MinIO
# Delta format automatically handles structure and data types flawlessly
df_geo = (
    spark.read
    .format("delta")
    .load("s3a://bronze/csv/geolocation/")
)

# 2. Preview the first 10 rows inside the notebook
display(df_geo.limit(10))

DataFrame[geolocation_zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, _ingested_at: timestamp, _source_file: string]

### **2. Schema Inspection**
Validates structural data types and column nullability to identify formatting requirements and ensure alignment with downstream relational components.

In [3]:
df_geo.printSchema()

root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



In [4]:
from pyspark.sql.types import StringType, IntegerType, DoubleType
from pyspark.sql import functions as F

# Final casting and schema enforcement for silver_geolocation
df_geo = df_geo \
    .withColumn("geolocation_zip_code_prefix", F.col("geolocation_zip_code_prefix").cast(IntegerType())) \
    .withColumn("geolocation_lat", F.col("geolocation_lat").cast(DoubleType())) \
    .withColumn("geolocation_lng", F.col("geolocation_lng").cast(DoubleType())) \
    .withColumn("geolocation_city", F.col("geolocation_city").cast(StringType())) \
    .withColumn("geolocation_state", F.col("geolocation_state").cast(StringType()))

# Verification of the final schema
print("=== Final Schema for silver_geolocation ===")
df_geo.printSchema()

# Preview a sample
print("=== Final Data Sample Preview ===")
df_geo.show(10, truncate=False)

=== Final Schema for silver_geolocation ===
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

=== Final Data Sample Preview ===
+---------------------------+-------------------+------------------+----------------+-----------------+--------------------------+-----------------------------+
|geolocation_zip_code_prefix|geolocation_lat    |geolocation_lng   |geolocation_city|geolocation_state|_ingested_at              |_source_file                 |
+---------------------------+-------------------+------------------+----------------+-----------------+--------------------------+-----------------------------+
|1037                       |-23.54562128115268 |-46.63929204800168|sao paulo       |SP    

### **3. Statistical Profiling & Outlier Detection**
Generates summary statistics (`min`, `max`, `mean`) to mathematically audit data distribution, capture data anomalies, and identify initial spatial outliers.

In [5]:
display(df_geo.describe())

DataFrame[summary: string, geolocation_zip_code_prefix: string, geolocation_lat: string, geolocation_lng: string, geolocation_city: string, geolocation_state: string, _source_file: string]

### **Key Insight from Profiling:**
* **Numeric Boundaries:** The initial statistical summary reveals coordinates that structurally deviate from Brazil's official geographical boundary box, indicating the presence of extreme spatial noise (outliers).


### **4. Redundancy & Duplicate Audit**
Evaluates dataset integrity by calculating the exact volume and ratio of duplicate rows. This identifies raw data replication

In [6]:
# Checking for redundant records within the Orders dataset
total_count = df_geo.count()
distinct_count = df_geo.distinct().count()

duplicate_count = total_count - distinct_count
print(f"Total count: {total_count}")
print(f"Total Duplicates: {duplicate_count}")
print(f"Duplicates Ratio: {(duplicate_count / total_count) * 100:.2f}%")

Total count: 1000163
Total Duplicates: 261831
Duplicates Ratio: 26.18%


## **Exact Row Deduplication & Spatial Integrity**

During the data quality audit of the Geolocation dataset, a high redundancy rate of approximately **26% exact row duplicates** was detected. 

### **Root Cause Analysis:**
This behavior is a known artifact of the source system's data collection process, where identical spatial profiles (matching Zip Code, Latitude, Longitude, City, and State) were logged repeatedly across multiple updates instead of enforcing strict primary key constraints.

### **Engineering Action:**
Since these exact redundant rows offer no additional business value and will trigger severe data inflation (Data Explosion) during downstream relational JOINs, a strict deduplication pipeline is enforced here. We apply `.dropDuplicates()` to ensure data uniqueness, optimize storage footprint, and protect the integrity of subsequent Lakehouse layers.

In [7]:
# Save the total count before cleaning for tracking purposes
initial_row_count = df_geo.count()

# Apply exact row deduplication across all columns
df_geo_clean = df_geo.dropDuplicates()

# Track the final clean count and display confirmation
final_row_count = df_geo_clean.count()
print(f"=== Geolocation Deduplication Complete ===")
print(f"Rows before deduplication: {initial_row_count}")
print(f"Rows after deduplication:  {final_row_count}")
print(f"Successfully dropped {initial_row_count - final_row_count} redundant rows.")

=== Geolocation Deduplication Complete ===
Rows before deduplication: 1000163
Rows after deduplication:  738332
Successfully dropped 261831 redundant rows.


In [8]:
display(df_geo_clean.describe())

DataFrame[summary: string, geolocation_zip_code_prefix: string, geolocation_lat: string, geolocation_lng: string, geolocation_city: string, geolocation_state: string, _source_file: string]

### **5. Data Completeness & Null Value Scan**
Executes a columnar scan across the dataset to detect and count missing values (`NULL`s), ensuring structural completeness.

In [9]:
# Analyzing data completeness by counting NULL values in each column of the Orders tablefrom pyspark.sql import functions as F
from pyspark.sql import functions as F
null_counts = df_geo_clean.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_geo_clean.columns])

display(null_counts)

DataFrame[geolocation_zip_code_prefix: bigint, geolocation_lat: bigint, geolocation_lng: bigint, geolocation_city: bigint, geolocation_state: bigint, _ingested_at: bigint, _source_file: bigint]

### **6. Categorical Cardinality & Distribution Analysis**
Profiles the unique value counts (cardinality) for cities and states, and examines record distribution across regions to understand geographical density.

In [10]:
from pyspark.sql import functions as F

# 1. Count of distinct cities and states
unique_cities_count = df_geo_clean.select("geolocation_city").distinct().count()
unique_states_count = df_geo_clean.select("geolocation_state").distinct().count()

print("=== GEOLOCATION DOMAIN SUMMARY ===")
print(f"Total Unique Cities: {unique_cities_count}")
print(f"Total Unique States: {unique_states_count}")
print("="*40)

# 2. Display the top 15 states with the most records to see the data distribution
print("\n=== STATES BY RECORD COUNT ===")
df_geo_clean.groupBy("geolocation_state") \
    .agg(F.count("*").alias("record_count")) \
    .orderBy(F.col("record_count").desc()) \
    .show(27)

# 3. Display a sample of unique cities
print("\n=== SAMPLE OF DISTINCT CITIES ===")
df_geo_clean.select("geolocation_city").distinct().limit(15).show()

=== GEOLOCATION DOMAIN SUMMARY ===
Total Unique Cities: 8011
Total Unique States: 27

=== STATES BY RECORD COUNT ===
+-----------------+------------+
|geolocation_state|record_count|
+-----------------+------------+
|               SP|      285976|
|               MG|      101353|
|               RJ|       78836|
|               RS|       48093|
|               PR|       45059|
|               SC|       30191|
|               BA|       27720|
|               GO|       15601|
|               PE|       13162|
|               ES|       12632|
|               CE|        9541|
|               MT|        9374|
|               DF|        9080|
|               MS|        8594|
|               PA|        8551|
|               MA|        6277|
|               PB|        4787|
|               RN|        4014|
|               PI|        3592|
|               AL|        3415|
|               TO|        2977|
|               SE|        2653|
|               RO|        2523|
|               AM|      

### **Key Insight: Text Formatting Noise Captured**
* **The Issue:** Initial inspection of unique cities reveals severe text formatting noise (inconsistent casing, trailing spaces, and Portuguese diacritical marks/accents). This artificially inflates city cardinality (e.g., treating "São Paulo" and "sao paulo" as different entities).
* **Next Steps:** Standardizing city text is required to unify names before mapping coordinates.

### **7. Text Standardization & Accent Stripping**
Applies text normalization techniques by converting city names to lowercase, trimming padding spaces, and stripping Portuguese diacritical marks (accents) to resolve basic semantic variance.

In [11]:
from pyspark.sql import functions as F

df_geo_text_clean = df_geo_clean.withColumn(
    "geolocation_city",
    F.trim(F.lower(F.col("geolocation_city")))
).withColumn(
    "geolocation_city",
    F.translate(F.col("geolocation_city"), "ãáâéíóôúç", "aaaeioouc") 
)

cleaned_cities_count = df_geo_text_clean.select("geolocation_city").distinct().count()
print(f"Unique Cities BEFORE Text Cleaning: {unique_cities_count}")
print(f"Unique Cities AFTER Text Cleaning:  {cleaned_cities_count}")

Unique Cities BEFORE Text Cleaning: 8011
Unique Cities AFTER Text Cleaning:  6054


### **Key Insight: Hidden Typos & Residual Noise**
* **The Progress:** Basic text standardization successfully reduced unique city cardinality from **8,011 to 6,054**.
* **The Remaining Issue:** Although improved, **6,054** is still artificially higher than Brazil's actual number of official municipalities (5,570). This indicates that hidden text noise (like complex spelling mistakes or structural typos attached to the same zip code) still exists and cannot be solved by simple string trimming.
* **Next Steps:** A dynamic window function voting mechanism is required to map and enforce the most frequent (dominant) clean city name per zip code prefix.

### **8. Advanced Windowing & Voting Mechanism for City Standardization**
Implements a dynamic window function to eliminate lingering typos. It groups data by zip code, counts city occurrences, ranks them using `row_number()`, and isolates the dominant (most frequent) clean city name per zip code prefix to act as the golden source truth.

In [12]:
from pyspark.sql import functions as F
from pyspark.sql import window as W

# A- Calculate the frequency of each city name per zip code prefix
city_counts = df_geo_text_clean.groupby("geolocation_zip_code_prefix", "geolocation_city").count()

# B- Define a window to rank cities within each zip code by frequency in descending order
window_spec = W.Window.partitionBy("geolocation_zip_code_prefix").orderBy(F.desc("count"))

# C- Assign a rank to each city per zip code, where the most frequent city is ranked 1
df_ranked_cities = city_counts.withColumn("rank", F.row_number().over(window_spec))

# D- Filter to retain only the top-ranked (most frequent) city for each zip code prefix
df_standardized_mapping = df_ranked_cities.filter(F.col("rank") == 1) \
    .select("geolocation_zip_code_prefix", F.col("geolocation_city").alias("standardized_city"))

# E- Join the standardized mapping back to the original table to replace inconsistent city names
df_geo_cities_fixed = df_geo_text_clean.join(df_standardized_mapping, on="geolocation_zip_code_prefix", how="inner")

print(f"Unique Cities BEFORE this step: {df_geo_text_clean.select('geolocation_city').distinct().count()}")
print(f"Unique Cities AFTER this step:  {df_geo_cities_fixed.select('standardized_city').distinct().count()}")

Unique Cities BEFORE this step: 6054
Unique Cities AFTER this step:  5773


### **Key Insight: Resolution of Semantic Inconsistencies**
* **The Progress:** The voting mechanism successfully streamlined unique city cardinality down from **6,054 to 5,768**.
* **Geographical Justification:** While **5,768** is still slightly higher than Brazil's official 5,570 major municipalities, this variance is completely normal and accurate. It accounts for smaller administrative districts, local sub-municipalities, and sovereign islands that share or partition identical zip code prefixes. 
* **Conclusion:** The textual names are now fully standardized and structurally clean.

In [13]:
display(df_geo_cities_fixed.limit(10))

DataFrame[geolocation_zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, _ingested_at: timestamp, _source_file: string, standardized_city: string]

### **9. Spatial Outlier Detection via Bounding Box Filtering**
Establishes a geospatial bounding box based on Brazil's official extreme coordinates. This isolates severe latitude/longitude noise (e.g., coordinates incorrectly pointing to Europe or Asia) to audit spatial data quality.

In [14]:
from pyspark.sql import functions as F

# 1. Define logical geographical boundaries for Brazil
brazil_lat_min, brazil_lat_max = -34.0, 6.0
brazil_lng_min, brazil_lng_max = -75.0, -28.0

# 2. Calculate total record count
total_rows = df_geo_cities_fixed.count()

# 3. Identify outlier rows (outside Brazil's bounding box)
df_spatial_errors = df_geo_cities_fixed.filter(
    (F.col("geolocation_lat") < brazil_lat_min) | (F.col("geolocation_lat") > brazil_lat_max) |
    (F.col("geolocation_lng") < brazil_lng_min) | (F.col("geolocation_lng") > brazil_lng_max)
)
error_count = df_spatial_errors.count()

# 4. Print spatial error analysis report
print("=== SPATIAL ERROR ANALYSIS ===")
print(f"Total Evaluated Rows: {total_rows}")
print(f"Total Outlier Rows (Outside Brazil): {error_count}")
print(f"Spatial Error Ratio: {(error_count / total_rows) * 100:.4f}%")
print("="*30)

df_spatial_errors.show(33)

=== SPATIAL ERROR ANALYSIS ===
Total Evaluated Rows: 738332
Total Outlier Rows (Outside Brazil): 27
Spatial Error Ratio: 0.0037%
+---------------------------+-------------------+-------------------+--------------------+-----------------+--------------------+--------------------+--------------------+
|geolocation_zip_code_prefix|    geolocation_lat|    geolocation_lng|    geolocation_city|geolocation_state|        _ingested_at|        _source_file|   standardized_city|
+---------------------------+-------------------+-------------------+--------------------+-----------------+--------------------+--------------------+--------------------+
|                      18243| 28.008978338034268|-15.536867177679131|bom retiro da esp...|               SP|2026-06-08 03:38:...|olist_geolocation...|bom retiro da esp...|
|                      28155|  42.43928591592116|  13.82021409761152|         santa maria|               RJ|2026-06-08 03:38:...|olist_geolocation...|         santa maria|
|          

### **Key Insight: Minimal but Critical Spatial Noise**
* **The Result:** Only **27 rows out of 738,332** were flagged as true spatial outliers, yielding an extremely low error ratio of **0.0037%**.
* **The Root Cause:** These 27 points possess corrupted GPS coordinates that place them completely outside the South American continent (mostly due to data insertion bugs or inverted signs).
* **Next Steps:** Instead of dropping these rows (which would cause missing values when joining with customers/sellers), a **Conditional Imputation** strategy will be applied. These 27 invalid coordinates will be replaced by the computed geographical mean of valid coordinates sharing the exact same zip code prefix.

### **10. Capturing Spatial Anomalies & Error Logging**
Appends a metadata column (`error_reason`) to the isolated spatial outliers and persists them into a dedicated log table (`silver_geolocation_spatial_errors`) to maintain a clear audit trail for data quality governance.

In [15]:
from pyspark.sql import functions as F

# 1. Append a spatial error reason column to the identified outliers
df_spatial_errors_logged = df_spatial_errors.withColumn(
    "error_reason", 
    F.lit("Spatial Outlier: Coordinates located outside Brazil's official geographic bounding box")
)

# 2. Persist the spatial errors into a dedicated audit table
df_spatial_errors_logged.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_geolocation_spatial_errors")

print("Spatial errors log table created successfully with error reasons!")

Spatial errors log table created successfully with error reasons!


### **11. Spatial Imputation via Zip-Code Centroids**
Resolves the 27 spatial outliers using a **Conditional Mean Substitution** strategy:

* **Step 1 (Compute Centroids):** Groups valid coordinates by `zip_code` to calculate the geographical **Mean Latitude & Longitude** for each region.
* **Step 2 (Conditional Patching):** Uses `F.when()` to replace **only** the 27 corrupted coordinates outside Brazil with their local calculated means.
* **Step 3 (Data Preservation):** Keeps all original, valid coordinates intact without dropping a single row or losing data granularity.

In [16]:
from pyspark.sql import functions as F

# 1. Calculate the mean coordinates for valid records per zip code prefix
# This acts as a repair dictionary for outlier correction
df_zip_means = df_geo_cities_fixed.filter(
    (F.col("geolocation_lat") >= brazil_lat_min) & (F.col("geolocation_lat") <= brazil_lat_max) &
    (F.col("geolocation_lng") >= brazil_lng_min) & (F.col("geolocation_lng") <= brazil_lng_max)
).groupBy("geolocation_zip_code_prefix") \
 .agg(
     F.mean("geolocation_lat").alias("mean_lat"),
     F.mean("geolocation_lng").alias("mean_lng")
 )

# 2. Join the calculated means with the original dataset
df_geo_with_means = df_geo_cities_fixed.join(df_zip_means, on="geolocation_zip_code_prefix", how="left")

# 3. Apply conditional imputation to correct spatial outliers
# Replace coordinates with mean values if they fall outside defined boundaries
df_geo_final_silver = df_geo_with_means.withColumn(
    "geolocation_lat",
    F.when(
        (F.col("geolocation_lat") < brazil_lat_min) | (F.col("geolocation_lat") > brazil_lat_max),
        F.col("mean_lat")
    ).otherwise(F.col("geolocation_lat"))
).withColumn(
    "geolocation_lng",
    F.when(
        (F.col("geolocation_lng") < brazil_lng_min) | (F.col("geolocation_lng") > brazil_lng_max),
        F.col("mean_lng")
    ).otherwise(F.col("geolocation_lng"))
)

# 4. Clean up auxiliary columns used for calculation
df_geo_final_silver = df_geo_final_silver.drop("mean_lat", "mean_lng")

### **12. Post-Imputation Quality Assurance & Validation**
Re-evaluates the processed DataFrame against Brazil's bounding box constraints to mathematically verify that all spatial anomalies have been successfully resolved.

In [17]:
# The following validation step confirms that all spatial outliers have been corrected
# by verifying that no records remain outside the defined geographic boundaries.
remaining_errors = df_geo_final_silver.filter(
    (F.col("geolocation_lat") < brazil_lat_min) | (F.col("geolocation_lat") > brazil_lat_max) |
    (F.col("geolocation_lng") < brazil_lng_min) | (F.col("geolocation_lng") > brazil_lng_max)
).count()

print(f"Remaining spatial outliers after conditional imputation: {remaining_errors}")

Remaining spatial outliers after conditional imputation: 0


### **13. Mapping State Abbreviations to Full Geographical Names**
Enriches the dataset by converting 2-letter state abbreviations (e.g., 'SP', 'RJ') into their official full Portuguese names 

In [18]:
from pyspark.sql import functions as F
from itertools import chain

# 1. Define a mapping dictionary for all 27 Brazilian states
brazil_states_map = {
    'SP': 'São Paulo',
    'MG': 'Minas Gerais',
    'RJ': 'Rio de Janeiro',
    'RS': 'Rio Grande do Sul',
    'PR': 'Paraná',
    'SC': 'Santa Catarina',
    'BA': 'Bahia',
    'GO': 'Goiás',
    'PE': 'Pernambuco',
    'ES': 'Espírito Santo',
    'CE': 'Ceará',
    'MT': 'Mato Grosso',
    'DF': 'Distrito Federal',
    'MS': 'Mato Grosso do Sul',
    'PA': 'Pará',
    'MA': 'Maranhão',
    'PB': 'Paraíba',
    'RN': 'Rio Grande do Norte',
    'PI': 'Piauí',
    'AL': 'Alagoas',
    'TO': 'Tocantins',
    'SE': 'Sergipe',
    'RO': 'Rondônia',
    'AM': 'Amazonas',
    'AC': 'Acre',
    'AP': 'Amapá',
    'RR': 'Roraima'
}

# 2. Convert dictionary into a Spark-compatible map expression
mapping_expr = F.create_map([F.lit(x) for x in chain(*brazil_states_map.items())])

# 3. Map abbreviations to full names
# Use coalesce to retain original abbreviation if a match is not found
df_geo_states_fixed = df_geo_final_silver.withColumn(
    "geolocation_state_full", 
    F.coalesce(mapping_expr[F.col("geolocation_state")], F.col("geolocation_state"))
)

# 4. Verification of the standardization process
print("=== STATES WITH FULL NAMES ===")
df_geo_states_fixed.groupBy("geolocation_state_full") \
    .agg(F.count("*").alias("record_count")) \
    .orderBy(F.col("record_count").desc()) \
    .show(27, truncate=False)

=== STATES WITH FULL NAMES ===
+----------------------+------------+
|geolocation_state_full|record_count|
+----------------------+------------+
|São Paulo             |285976      |
|Minas Gerais          |101353      |
|Rio de Janeiro        |78836       |
|Rio Grande do Sul     |48093       |
|Paraná                |45059       |
|Santa Catarina        |30191       |
|Bahia                 |27720       |
|Goiás                 |15601       |
|Pernambuco            |13162       |
|Espírito Santo        |12632       |
|Ceará                 |9541        |
|Mato Grosso           |9374        |
|Distrito Federal      |9080        |
|Mato Grosso do Sul    |8594        |
|Pará                  |8551        |
|Maranhão              |6277        |
|Paraíba               |4787        |
|Rio Grande do Norte   |4014        |
|Piauí                 |3592        |
|Alagoas               |3415        |
|Tocantins             |2977        |
|Sergipe               |2653        |
|Rondônia          

### **14. Final Schema Realignment**
Cleans up the final schema to deliver a production-ready DataFrame:
* **Drops:** Old, noisy `geolocation_city` and abbreviation `geolocation_state` columns.
* **Renames:** Restores the newly standardized columns back to their original names (`geolocation_city` and `geolocation_state`).
* **Result:** A clean, optimized schema ready for the Silver layer without any redundant data.

In [19]:
# 1. Remove legacy columns (abbreviations and unstandardized city names)
df_geo_final_silver = df_geo_states_fixed.drop("geolocation_state", "geolocation_city")

# 2. Rename standardized columns to align with the original production schema
df_geo_final_silver = df_geo_final_silver \
    .withColumnRenamed("standardized_city", "geolocation_city") \
    .withColumnRenamed("geolocation_state_full", "geolocation_state")

In [21]:
from pyspark.sql import Row

# 1. Define appropriate placeholder values
# Use -1 for zip code, 0.0 for coordinates, and "Unknown" for descriptive fields
placeholder_data = [
    Row(
        geolocation_zip_code_prefix=-1,
        geolocation_lat=0.0,
        geolocation_lng=0.0,
        geolocation_city="Unknown City",
        geolocation_state="Unknown State"
    )
]

# 2. Create a DataFrame containing only the placeholder record
df_placeholder = spark.createDataFrame(placeholder_data)

# 3. FIXED: Use unionByName with allowMissingColumns=True to seamlessly merge schemas
# This automatically handles the mismatch in column counts (7 vs 5) by filling missing fields with null
df_geo_final_silver = df_geo_final_silver.unionByName(df_placeholder, allowMissingColumns=True)

# 4. Remove duplicate entries for the placeholder ID to ensure uniqueness
df_geo_final_silver = df_geo_final_silver.dropDuplicates(["geolocation_zip_code_prefix"])

print("Placeholder row added successfully.")

Placeholder row added successfully.


In [22]:
display(df_geo_final_silver.limit(10))

DataFrame[geolocation_zip_code_prefix: bigint, geolocation_lat: double, geolocation_lng: double, _ingested_at: timestamp, _source_file: string, geolocation_city: string, geolocation_state: string]

In [23]:
from pyspark.sql.types import StringType, IntegerType, DoubleType
from pyspark.sql import functions as F

# Final casting and schema enforcement for silver_geolocation
df_geo_final_silver = df_geo_final_silver \
    .withColumn("geolocation_zip_code_prefix", F.col("geolocation_zip_code_prefix").cast(IntegerType())) \
    .withColumn("geolocation_lat", F.col("geolocation_lat").cast(DoubleType())) \
    .withColumn("geolocation_lng", F.col("geolocation_lng").cast(DoubleType())) \
    .withColumn("geolocation_city", F.col("geolocation_city").cast(StringType())) \
    .withColumn("geolocation_state", F.col("geolocation_state").cast(StringType()))

# Verification of the final schema
print("=== Final Schema for silver_geolocation ===")
df_geo_final_silver.printSchema()

# Preview a sample
print("=== Final Data Sample Preview ===")
df_geo_final_silver.show(10, truncate=False)

=== Final Schema for silver_geolocation ===
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)

=== Final Data Sample Preview ===
+---------------------------+-------------------+-------------------+--------------------------+-----------------------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat    |geolocation_lng    |_ingested_at              |_source_file                 |geolocation_city|geolocation_state|
+---------------------------+-------------------+-------------------+--------------------------+-----------------------------+----------------+-----------------+
|-1                         |0.0                |0.0                |NULL               

### **15. Silver Layer Data Persistence**
Persists the final cleaned dataset into the Silver Layer as a managed **Delta Table** for SQL operations and as **Parquet Files** for storage portability.

In [24]:
# ==========================================================================
# FINAL PERSISTENCE: SAVING REFINED SILVER GEOLOCATION
# ==========================================================================

# Define the target absolute storage paths on MinIO
SILVER_GEOLOCATION_DELTA_PATH   = "s3a://silver/refined/geolocation/"
SILVER_GEOLOCATION_PARQUET_PATH = "s3a://silver/refined/geolocation_parquet/"

# FIX FOR MINIO/DELTA: Drop ambiguous metadata columns to prevent duplicate columns error during save
df_geo_final_silver_cleaned = df_geo_final_silver.drop("_ingested_at", "_source_file")

# 1. Save as a Delta Table using direct MinIO S3A paths (FIXED: replaced saveAsTable)
df_geo_final_silver_cleaned.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(SILVER_GEOLOCATION_DELTA_PATH)

# Refresh the Delta cache for immediate query capability inside the cluster
spark.catalog.refreshByPath(SILVER_GEOLOCATION_DELTA_PATH)


# 2. Export as Parquet files to the Silver directory on MinIO (FIXED: updated local path to S3A)
df_geo_final_silver_cleaned.write \
    .mode("overwrite") \
    .parquet(SILVER_GEOLOCATION_PARQUET_PATH)

print("Success! Geolocation Table is successfully saved to MinIO as Delta and exported as Parquet files!")

Success! Geolocation Table is successfully saved to MinIO as Delta and exported as Parquet files!
